# Stage 3 — DPO Preference Alignment

This notebook continues from the SFT adapter and aligns it on 66 preference pairs. Each chosen answer is safe and policy-grounded; each rejected answer is generic, incomplete, or unsafe.

In [ ]:
# Run on a Google Colab GPU runtime (T4 or better).
!pip -q install -U unsloth trl transformers datasets peft accelerate bitsandbytes sentencepiece

In [ ]:
from pathlib import Path
import json, os, sys, torch

# Expected workflow: clone the GitHub repository, then open the notebook from the repo.
CANDIDATES = [Path.cwd(), Path('/content/domain-ai-assistant-finetuning')]
ROOT = next((p for p in CANDIDATES if (p / 'data').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Repository root not found. Clone the repo into /content/domain-ai-assistant-finetuning or run from the repo root.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Repository:', ROOT)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU runtime is required for these notebooks.')

In [ ]:
from datasets import Dataset
from peft import PeftModel
from trl import DPOConfig, DPOTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported
from common import SYSTEM_PROMPT, generate_answer, read_jsonl, write_comparison_report

BASE_MODEL = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 1024
SFT_ADAPTER = ROOT / 'outputs/sft_adapter'
DPO_ADAPTER = ROOT / 'outputs/dpo_adapter'
if not SFT_ADAPTER.exists():
    raise FileNotFoundError('Run instruction_finetuning.ipynb first.')

## 1. Load and format the preference dataset

In [ ]:
rows = read_jsonl(ROOT / 'data/preference_dataset.jsonl')
assert len(rows) >= 50
assert all(r['chosen'] != r['rejected'] for r in rows)
formatted = []
for row in rows:
    formatted.append({
        'prompt': [
            {'role':'system', 'content':SYSTEM_PROMPT},
            {'role':'user', 'content':row['prompt']},
        ],
        'chosen': [{'role':'assistant', 'content':row['chosen']}],
        'rejected': [{'role':'assistant', 'content':row['rejected']}],
    })
dataset = Dataset.from_list(formatted).train_test_split(test_size=0.1, seed=42)
print(dataset)
print(dataset['train'][0])

## 2. Reload the SFT policy as a trainable PEFT model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
model = PeftModel.from_pretrained(model, str(SFT_ADAPTER), is_trainable=True)
model.print_trainable_parameters()

## 3. Run Direct Preference Optimization

In [ ]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    processing_class=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    args=DPOConfig(
        output_dir=str(ROOT / 'outputs/dpo_training'),
        max_length=MAX_SEQ_LENGTH,
        beta=0.1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        learning_rate=1e-5,
        warmup_ratio=0.05,
        logging_steps=2,
        eval_strategy='epoch',
        save_strategy='epoch',
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        seed=42,
        report_to='none',
    ),
)
train_result = trainer.train()
print(train_result)
(ROOT / 'artifacts/dpo_train_metrics.json').write_text(
    json.dumps(train_result.metrics, indent=2, default=str), encoding='utf-8'
)
trainer.save_state()

## 4. Save and generate the final three-stage evaluation

In [ ]:
DPO_ADAPTER.mkdir(parents=True, exist_ok=True)
model.save_pretrained(DPO_ADAPTER)
tokenizer.save_pretrained(DPO_ADAPTER)
FastLanguageModel.for_inference(model)
questions = json.loads((ROOT / 'data/evaluation_questions.json').read_text())
dpo_answers = [generate_answer(model, tokenizer, q) for q in questions]
(ROOT / 'artifacts/dpo_outputs.json').write_text(json.dumps(dpo_answers, indent=2), encoding='utf-8')

def load_or_placeholder(name):
    path = ROOT / f'artifacts/{name}_outputs.json'
    return json.loads(path.read_text()) if path.exists() else [f'Run {name} evaluation'] * len(questions)

base_answers = load_or_placeholder('base')
sft_answers = load_or_placeholder('sft')
write_comparison_report(
    ROOT / 'reports/final_evaluation.md',
    'Final Evaluation: Base vs SFT vs DPO',
    questions,
    [
        ('Base model answer', base_answers),
        ('SFT model answer', sft_answers),
        ('DPO model answer', dpo_answers),
        ('Best answer', ['Complete after human review'] * len(questions)),
        ('Reason', ['Compare correctness, helpfulness, domain accuracy, safety, tone, clarity, and hallucination risk'] * len(questions)),
    ],
)
print('Saved adapter:', DPO_ADAPTER)
print('Updated reports/final_evaluation.md')

## Final review

Edit the last column of `reports/final_evaluation.md`. Select the best stage and explain the decision using correctness, helpfulness, domain accuracy, safety, tone, clarity, hallucination reduction, and professional response quality. Capture the DPO training metrics, especially the chosen/rejected reward margin if logged.